In [9]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import sqlite3

In [10]:
conn = sqlite3.connect("wine_reviews.db")

In [12]:
varieties = pd.read_sql_query(
    "SELECT variety FROM varieties ORDER BY variety;", conn
)

print(varieties.nunique(), "Variety... Count")
varieties.head(10).T

variety    756
dtype: int64 Variety... Count


,0,1,2,3,4,5,6,7,8,9
variety,Abouriou,Agiorgitiko,Aglianico,Aidani,Airen,Albana,Albanello,Albariño,Albarossa,Albarín


In [13]:
variety_pct = pd.read_sql_query("""
    SELECT v.variety,
           COUNT(*) AS num_reviews,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM reviews r
    JOIN varieties v ON r.variety_id = v.variety_id
    GROUP BY v.variety
    ORDER BY pct DESC;
""", conn)

print(len(variety_pct), "varieties")
variety_pct.head(20)

756 varieties


,variety,num_reviews,pct
0,Pinot Noir,16658,9.82
1,Chardonnay,15640,9.22
2,Cabernet Sauvignon,13274,7.83
3,Red Blend,11226,6.62
4,Bordeaux-style Red Blend,9008,5.31
5,Sauvignon Blanc,6815,4.02
6,Riesling,6614,3.90
7,Syrah,5863,3.46
8,Merlot,4757,2.80
9,Rosé,3920,2.31


In [14]:
variety_pct.to_csv("variety_percentages.csv", index=False, encoding="utf-8-sig")

In [15]:
all_reviews = pd.read_sql_query("""
    SELECT r.review_id,
           l.country,
           l.province,
           l.region_1,
           l.region_2,
           w.winery,
           v.variety,
           r.designation,
           r.points,
           r.price,
           r.description
    FROM reviews r
    LEFT JOIN wineries  w ON r.winery_id   = w.winery_id
    LEFT JOIN varieties v ON r.variety_id  = v.variety_id
    LEFT JOIN locations l ON r.location_id = l.location_id
    ORDER BY r.review_id;
""", conn)

print(all_reviews.shape)   # should match the row count of the reviews table
all_reviews.head()


(169613, 11)


,review_id,country,province,region_1,region_2,winery,variety,designation,points,price,description
0,1,Italy,Sicily & Sardinia,Etna,NaN,Nicosia,White Blend,Vulkà Bianco,87,NaN,"Aromas include tropical fruit, broom, brimston..."
1,2,Portugal,Douro,NaN,NaN,Quinta dos Avidagos,Portuguese Red,Avidagos,87,15.0,"This is ripe and fruity, a wine that is smooth..."
2,3,US,Oregon,Willamette Valley,Willamette Valley,Rainstorm,Pinot Gris,NaN,87,14.0,"Tart and snappy, the flavors of lime flesh and..."
3,4,US,Michigan,Lake Michigan Shore,NaN,St. Julian,Riesling,Reserve Late Harvest,87,13.0,"Pineapple rind, lemon pith and orange blossom ..."
4,5,US,Oregon,Willamette Valley,Willamette Valley,Sweet Cheeks,Pinot Noir,Vintner's Reserve Wild Child Block,87,65.0,"Much like the regular bottling from 2012, this..."
